# PlaceMux · AI/ML Matching Engine
## Exploration & Experiment Walkthrough
**Phase 2 · Week 3 · Quality Baseline Pipeline**

This notebook walks through:
1. Data generation & EDA
2. Feature engineering (F1–F5)
3. Baseline establishment
4. Model training + optimization
5. Full metric evaluation
6. SHAP explainability
7. One-example walkthrough

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from data.generate_data import generate_dataset
from utils.features import FEATURE_COLS
from models.baseline import SkillOverlapBaseline
from models.evaluate import evaluate_model, ndcg_at_k

print('All imports OK')

## 1. Generate & inspect data

In [ ]:
df = generate_dataset(n_candidates=600, n_jobs=200, pairs_per_candidate=4)
print(df.shape)
df.head()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, FEATURE_COLS):
    df[col].hist(ax=ax, bins=30, color='#4F46E5', alpha=0.7)
    ax.set_title(col)
    ax.set_xlabel('Value')
plt.suptitle('Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Label balance: {df['label'].value_counts().to_dict()}")

## 2. Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df[FEATURE_COLS]
y = df['label']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.40, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

## 3. Baseline

In [ ]:
baseline = SkillOverlapBaseline()
b_m = baseline.evaluate(X_test, y_test)
print('Baseline metrics:')
for k, v in b_m.items(): print(f'  {k}: {v}')

## 4. XGBoost (Best Model)

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_tr_bal, y_tr_bal = sm.fit_resample(X_train, y_train)

xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', verbosity=0)
cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

params = {'n_estimators':[100,200], 'max_depth':[3,5], 'learning_rate':[0.05,0.1]}
gs = GridSearchCV(xgb, params, cv=cv, scoring='f1', n_jobs=-1)
gs.fit(X_tr_bal, y_tr_bal, eval_set=[(X_val, y_val)], verbose=False)

best_xgb = gs.best_estimator_
print(f'Best params: {gs.best_params_}')

## 5. Full Evaluation

In [ ]:
from models.evaluate import evaluate_model, print_report
metrics = evaluate_model(best_xgb, X_test, y_test, threshold=0.42)
print_report(metrics, 'XGBoost (tuned)')

## 6. SHAP Global Feature Importance

In [ ]:
import shap
explainer = shap.TreeExplainer(best_xgb)
sv = explainer.shap_values(X_test)
if isinstance(sv, list): sv = sv[1]
shap.summary_plot(sv, X_test, feature_names=FEATURE_COLS, plot_type='bar')

## 7. One-Example Walkthrough

In [ ]:
# Pick a single candidate–job pair
example = X_test.iloc[[0]]
score   = best_xgb.predict_proba(example)[:, 1][0]

sv_single = explainer.shap_values(example)
if isinstance(sv_single, list): sv_single = sv_single[1]

print(f'Input features:')
print(example.to_string())
print(f'\nMatch score: {score:.4f}')
print(f'Verdict: {"Strong match" if score >= 0.75 else "Partial match" if score >= 0.50 else "Weak match"}')
print(f'\nSHAP attribution:')
for f, v in zip(FEATURE_COLS, sv_single[0]):
    bar = '█' * int(abs(v) * 50)
    print(f'  {f:<28}: {v:+.4f}  {bar}')